# A tiny protein language model learns contacts

*bioinformatics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/bioinformatics/protein_language_model)

A two-layer transformer with einsum attention, $A_{nhij} = \mathrm{softmax}_j(Q_{nhik}K_{nhjk}/\sqrt{d_k})$, is trained only to fill in masked letters of sequences from a Potts model with couplings planted on a lattice fold. Nobody tells it the structure, yet its symmetrised, APC-corrected attention maps point at the planted contacts, the idea behind the contact heads of protein language models.

**Vector form**

$$\mathbf A^{(h)} = \operatorname{softmax}\!\Big(\frac{(\mathbf X\mathbf W_Q^{(h)})(\mathbf X\mathbf W_K^{(h)})^{\mathsf T}}{\sqrt{d_k}}\Big),\qquad \mathbf X \leftarrow \mathbf X + \sum_h \mathbf A^{(h)}\mathbf X\mathbf W_V^{(h)}\mathbf W_O^{(h)},\qquad \mathcal L = -\!\!\sum_{i\in\text{masked}}\!\! \log p_\theta(s_i \mid \mathbf s_{\setminus\text{masked}})$$

**Index (tensor) form**

$$A_{nhij} = \mathrm{softmax}_j\Big(\frac{X_{nic}W^Q_{hck}\;X_{njc'}W^K_{hc'k}}{\sqrt{d_k}}\Big) \qquad X_{nic} \mathrel{+}= A_{nhij}\,X_{njc'}W^V_{hc'k}\,W^O_{hkc} \qquad C_{ij} = \tfrac12\big(\bar A_{lhij} + \bar A_{lhji}\big) - \text{APC}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $J_{iajb}$ | `planted_potts(contacts, q, coupling, ...)` | (L, q, L, q) | random coupling blocks only between residues that touch in the lattice fold; Gibbs sampling turns them into a sequence family |
| $X_{nic}$ | `w['tok'][tokens] + w['pos']` | (N, L, d) | token plus position embedding of letter $s_{ni}$ (masked letters use the [MASK] token) |
| $Q_{nhik} = X_{nic}W^Q_{hck}$ | `einsum('nic,hck->nhik', X, Wq)` | (N, H, L, d_k) | repeated $c$ ⇒ summed: queries of every head (keys and values alike) |
| $A_{nhij}$ | `softmax(einsum('nhik,nhjk->nhij', Q, K) / sqrt(dk), -1)` | (N, H, L, L) | repeated $k$ ⇒ summed; each row is a probability distribution over the attended positions $j$ |
| $A_{nhij}V_{nhjk}W^O_{hkc}$ | `einsum('nhik,hkc->nic', einsum('nhij,nhjk->nhik', A, V), Wo)` | (N, L, d) | repeated $j$, then $h, k$ ⇒ summed: what each position collects from the others |
| $\mathcal L$ | `masked_lm_loss(logits, seqs, mask)` | () | cross-entropy of the true letter at the 15 % masked positions, minimised with Adam |
| $C_{ij}$ | `apc(0.5 * (A + A.T)).mean((0, 1))` | (L, L) | attention averaged over test sequences, symmetrised and APC-corrected per head, then averaged over layers $l$ and heads $h$ |

Why does attention find contacts? In a Potts model the letter at site $i$, given all the others, depends only on the sites coupled to it (its Markov blanket), so the best way to guess a masked letter is to look at its contact partners, and attention learns to do just that. Rao et al., "Transformer protein language models are unsupervised structure learners" (ICLR 2021), showed that attention maps of large protein language models trained on millions of natural sequences encode contacts, read out with a logistic regression over the APC-corrected, symmetrised maps fitted on a handful of proteins. Later models such as ESM-2 (2022–2023) went on to predict 3D structure from such representations. Here the readout is fully unsupervised (a plain average over heads), the family is synthetic, and the test loss is compared with two context-free baselines: the unigram (one letter distribution) and the per-site profile. Beating the profile means the model uses context, i.e. the couplings. The serpentine fold puts contacts on anti-diagonals, like the strands of a β-sheet; random pairs would be right only about 5 % of the time. APC removes the background of positions that everybody attends to. Try weaker couplings, fewer sequences or a single head.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/bioinformatics/protein_language_model/main.py
"""Main loop: planted Potts family → Gibbs sampling → masked-LM training → contacts from attention → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors

ROWS, COLS = 5, 6  # lattice fold: L = 30 residues
N_TEST = 500


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    q = params["q"]
    coords = tensors.lattice_fold(ROWS, COLS, dev)  # (L, 2)
    contacts = tensors.contact_map(coords)  # (L, L) bool
    L, n_contacts = coords.shape[0], int(torch.triu(contacts, 1).sum())
    h, J = tensors.planted_potts(contacts, q, params["coupling"], 0.5, seed=0)

    n_seq = 800 if fast else params["n_seq"]
    seqs = solve.gibbs_sample(h, J, tensors.parity(coords), n_seq + N_TEST, 20 if fast else 120, seed=0)  # (N, L)
    train, test = seqs[:n_seq], seqs[n_seq:]
    ce_uni, ce_prof = solve.baselines(train, test, q)
    rand = solve.random_precision(contacts)
    print(
        f"family: {n_seq} training + {N_TEST} test sequences, L = {L}, q = {q} letters, {n_contacts} planted contacts"
    )
    print(f"baselines on test letters: unigram {ce_uni:.3f} nats, per-site profile {ce_prof:.3f} nats")

    torch.manual_seed(0)
    layers, heads, d = params["layers"], params["heads"], params["d_model"]
    w = tensors.init_weights(L, q, d, heads, layers, seed=0, device=dev)
    n_params = sum(t.numel() for t in tensors.parameters(w))
    n_steps = 60 if fast else params["steps"]
    print(f"transformer: {layers} layer(s) × {heads} heads, d = {d}, {n_params} weights")
    hist = solve.train(train, test, contacts, w, q, n_steps, 128, params["lr"], params["mask_rate"],
                       n_snapshots=6 if fast else 40)  # fmt: skip
    for k in range(0, len(hist["at"]), max(1, len(hist["at"]) // 6)):
        print(
            f"  step {hist['at'][k]:5d}  test loss {hist['test'][k]:.3f}  contact precision {hist['precision'][k]:.2f}"
        )
    print(f"final: test loss {hist['test'][-1]:.3f} nats (profile {ce_prof:.3f}), ", end="")
    print(f"precision@{n_contacts} = {hist['precision'][-1]:.2f} (random {rand:.3f})")

    with torch.no_grad():
        _, maps = equation.forward(test[:8], w)
    print(f"attention rows sum to 1: max |Σ_j A_ij − 1| = {(maps.sum(-1) - 1).abs().max():.1e}")
    return plot.animate(hist, contacts, n_contacts, ce_uni, ce_prof, rand, out_dir / "protein_language_model.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/bioinformatics/protein_language_model/equation.py
r"""Masked language model → tensor translation.

Data: sequences s_{ni} (n = sequence, i = position) drawn from a Potts model with couplings only on contacts,

    P(s) \propto \exp\big(h_i(s_i) + \tfrac12 J_{ij}(s_i, s_j)\big),   local field  E_{nia} = h_{ia} + J_{iajb} x_{njb}.

Model: embeddings X_{nic} = T_{t_{ni} c} + P_{ic}, then per layer (c = channel, h = head, k = channel in a head)

    Q_{nhik} = X_{nic} W^Q_{hck},   K_{nhjk} = X_{njc} W^K_{hck},   V_{nhjk} = X_{njc} W^V_{hck}     (c summed)
    A_{nhij} = \mathrm{softmax}_j\big(Q_{nhik} K_{nhjk} / \sqrt{d_k}\big)                              (k summed)
    X_{nic} \leftarrow X_{nic} + A_{nhij} V_{nhjk} W^O_{hkc}                                            (h, j, k summed)

and a small MLP; logits z_{nia} = X_{nic} W_{ca}. Training minimises the masked-token cross-entropy

    \mathcal L = -\frac{1}{|M|}\sum_{(n,i)\in M} \log \mathrm{softmax}_a(z_{nia})[s_{ni}].

Contacts from attention (Rao et al. 2021): average A over sequences, symmetrise, remove the background (APC):

    \bar A_{lhij} = \langle A_{nlhij}\rangle_n,  C_{ij} = \tfrac12(\bar A + \bar A^\top)_{ij},
    C^{APC}_{ij} = C_{ij} - C_{i\cdot}C_{\cdot j}/C_{\cdot\cdot}.
"""

import math

import torch

# --- the Potts model that generates the "protein family" ---


def local_field(x: torch.Tensor, h: torch.Tensor, J: torch.Tensor) -> torch.Tensor:
    """E_{nia} = h_{ia} + J_{iajb} x_{njb}: log-weight of letter a at site i given the others.  j, b summed."""
    return h + torch.einsum("iajb,njb->nia", J, x)


# --- the transformer ---


def layer_norm(X: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    """Normalise every token vector to zero mean and unit variance over its channels c (no learned scale)."""
    mu = X.mean(-1, keepdim=True)
    return (X - mu) / torch.sqrt(X.var(-1, unbiased=False, keepdim=True) + eps)


def embed(tokens: torch.Tensor, w: dict) -> torch.Tensor:
    """X_{nic} = T_{t_{ni} c} + P_{ic}: token plus position embedding.  tokens (N, L) → (N, L, d)."""
    return w["tok"][tokens] + w["pos"]


def attention_weights(X: torch.Tensor, Wq: torch.Tensor, Wk: torch.Tensor) -> torch.Tensor:
    r"""A_{nhij} = softmax_j(Q_{nhik} K_{nhjk} / \sqrt{d_k}).   X (N, L, d) → (N, H, L, L); each row sums to 1."""
    Q = torch.einsum("nic,hck->nhik", X, Wq)
    K = torch.einsum("njc,hck->nhjk", X, Wk)
    scores = torch.einsum("nhik,nhjk->nhij", Q, K) / math.sqrt(Q.shape[-1])
    return torch.softmax(scores, dim=-1)


def attend(X: torch.Tensor, A: torch.Tensor, Wv: torch.Tensor, Wo: torch.Tensor) -> torch.Tensor:
    """Δ_{nic} = A_{nhij} V_{nhjk} W^O_{hkc}: each position collects a weighted mix of the others.  h, j, k summed."""
    V = torch.einsum("njc,hck->nhjk", X, Wv)
    mix = torch.einsum("nhij,nhjk->nhik", A, V)  # j summed: the weighted mix of values
    return torch.einsum("nhik,hkc->nic", mix, Wo)  # h, k summed: heads merged back into d channels


def block(X: torch.Tensor, p: dict) -> tuple[torch.Tensor, torch.Tensor]:
    """One pre-norm transformer layer: X + attention(LN X), then X + MLP(LN X). Returns (X, A)."""
    Xn = layer_norm(X)
    A = attention_weights(Xn, p["Wq"], p["Wk"])
    X = X + attend(Xn, A, p["Wv"], p["Wo"])
    X = X + torch.relu(layer_norm(X) @ p["W1"]) @ p["W2"]
    return X, A


def forward(tokens: torch.Tensor, w: dict) -> tuple[torch.Tensor, torch.Tensor]:
    """Logits z_{nia} (N, L, q) and the attention of every layer (N, layers, H, L, L)."""
    X = embed(tokens, w)
    maps = []
    for p in w["layers"]:
        X, A = block(X, p)
        maps.append(A)
    return layer_norm(X) @ w["out"], torch.stack(maps, dim=1)


def masked_lm_loss(logits: torch.Tensor, seqs: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Mean cross-entropy (nats) of the true letter at the masked positions only."""
    logp = torch.log_softmax(logits, dim=-1)  # (N, L, q)
    true = torch.gather(logp, -1, seqs[..., None])[..., 0]  # (N, L) log-probability of the true letter
    return -(true * mask).sum() / mask.sum()


# --- contacts from attention ---


def apc(F: torch.Tensor) -> torch.Tensor:
    """Average-product correction F_ij − F_i· F_·j / F_·· (works on (..., L, L) stacks)."""
    row = F.mean(-1, keepdim=True)
    col = F.mean(-2, keepdim=True)
    return F - row * col / F.mean((-2, -1), keepdim=True)


def attention_contacts(maps: torch.Tensor) -> torch.Tensor:
    """Unsupervised contact score from attention: (N, layers, H, L, L) → (L, L).

    Average over sequences, symmetrise each head, zero the diagonal, APC per head, then average all heads.
    """
    A = maps.mean(0)  # (layers, H, L, L)
    C = 0.5 * (A + A.transpose(-1, -2))
    C = C * (1 - torch.eye(C.shape[-1], device=C.device))
    return apc(C).mean((0, 1))


def top_pairs(score: torch.Tensor, n_top: int, min_sep: int = 3) -> torch.Tensor:
    """The n_top highest-scoring pairs (i < j, j − i ≥ min_sep) as an (n_top, 2) index tensor."""
    L = score.shape[0]
    i, j = torch.triu_indices(L, L, min_sep, device=score.device)
    order = torch.argsort(score[i, j], descending=True)[:n_top]
    return torch.stack([i[order], j[order]], dim=1)


def precision(pairs: torch.Tensor, contacts: torch.Tensor) -> float:
    """Fraction of predicted pairs that are true contacts."""
    return contacts[pairs[:, 0], pairs[:, 1]].float().mean().item()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/bioinformatics/protein_language_model/tensors.py
"""Build the concrete tensors: a 2D lattice fold and its contact map, the planted Potts model, the sequences
and the transformer weights.

Letters are integers 0 … q−1; the extra token q is [MASK]. The weights are plain tensors in a dict (no nn.Module)
so every product in equation.py is an explicit einsum.
"""

import math

import torch


def lattice_fold(rows: int, cols: int, device: torch.device) -> torch.Tensor:
    """A serpentine chain on a rows × cols square lattice: residue i sits at r_{ia}, shape (L, 2), L = rows·cols.

    Rows run left-to-right, then right-to-left, … so neighbouring rows pair up like the strands of a β-sheet.
    """
    coords = []
    for y in range(rows):
        xs = range(cols) if y % 2 == 0 else range(cols - 1, -1, -1)
        coords += [(x, y) for x in xs]
    return torch.tensor(coords, device=device)


def contact_map(coords: torch.Tensor, min_sep: int = 3) -> torch.Tensor:
    """True contacts (L, L) bool: lattice neighbours that are at least min_sep apart along the chain."""
    dist = (coords[:, None, :] - coords[None, :, :]).abs().sum(-1)  # city-block distance (L, L)
    idx = torch.arange(coords.shape[0], device=coords.device)
    return (dist == 1) & ((idx[:, None] - idx[None, :]).abs() >= min_sep)


def parity(coords: torch.Tensor) -> torch.Tensor:
    """Checkerboard colour (L,) bool. Lattice neighbours always have opposite colours."""
    return coords.sum(-1) % 2 == 0


def planted_potts(contacts: torch.Tensor, q: int, coupling: float, field: float, seed: int):
    """Potts parameters with couplings ONLY on the planted contacts.

    h_{ia} (L, q) ~ field · N(0, 1);  J_{iajb} (L, q, L, q): a random q×q block ~ coupling · N(0, 1) per contact,
    with J_{iajb} = J_{jbia} and zero everywhere else.
    """
    gen = torch.Generator().manual_seed(seed)
    L, dev = contacts.shape[0], contacts.device
    h = field * torch.randn(L, q, generator=gen)
    J = torch.zeros(L, q, L, q)
    for i, j in torch.nonzero(torch.triu(contacts.cpu(), 1)).tolist():
        block = coupling * torch.randn(q, q, generator=gen)
        J[i, :, j, :] = block
        J[j, :, i, :] = block.T
    return h.to(dev), J.to(dev)


def one_hot(seqs: torch.Tensor, q: int) -> torch.Tensor:
    """Integer sequences (N, L) → one-hot x_{nia} (N, L, q), float."""
    return torch.nn.functional.one_hot(seqs, q).float()


def init_weights(L: int, q: int, d: int, heads: int, layers: int, seed: int, device: torch.device) -> dict:
    """Transformer weights. c = model channel (d of them), h = head, k = channel within a head (d / heads).

    tok (q+1, d)  token embedding (last row = [MASK]);  pos (L, d)  learned position embedding
    per layer l:  Wq, Wk, Wv (H, d, d/H),  Wo (H, d/H, d),  W1 (d, 2d), W2 (2d, d)
    out (d, q)    read-out to letter logits
    """
    gen = torch.Generator().manual_seed(seed)
    dk = d // heads

    def rand(*shape, scale):
        return (scale * torch.randn(*shape, generator=gen)).to(device).requires_grad_(True)

    s = 1 / math.sqrt(d)  # keeps activations O(1) at the start
    w = {"tok": rand(q + 1, d, scale=0.5), "pos": rand(L, d, scale=0.5), "out": rand(d, q, scale=s)}
    w["layers"] = [
        {
            "Wq": rand(heads, d, dk, scale=s),
            "Wk": rand(heads, d, dk, scale=s),
            "Wv": rand(heads, d, dk, scale=s),
            "Wo": rand(heads, dk, d, scale=s),
            "W1": rand(d, 2 * d, scale=s),
            "W2": rand(2 * d, d, scale=s / math.sqrt(2)),
        }
        for _ in range(layers)
    ]
    return w


def parameters(w: dict) -> list[torch.Tensor]:
    """All trainable tensors, for the optimiser."""
    return [w["tok"], w["pos"], w["out"]] + [t for layer in w["layers"] for t in layer.values()]


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/bioinformatics/protein_language_model/solve.py
"""Sample a protein family from the planted Potts model, then train the masked language model on it."""

import math

import torch

from problems._common.progress import steps

from . import equation, tensors


def gibbs_sample(h, J, colour, n_seq: int, n_sweeps: int, seed: int) -> torch.Tensor:
    """N independent Markov chains updated together; returns the final sequences (N, L).

    Checkerboard Gibbs: contacts only join sites of opposite colour, so all sites of one colour are independent given
    the rest and are redrawn at once from softmax_a E_{nia} (Gumbel-max trick: argmax_a E_{nia} − log(−log U)).
    """
    gen = torch.Generator(device=h.device).manual_seed(seed)
    L, q = h.shape
    s = torch.randint(q, (n_seq, L), generator=gen, device=h.device)
    for _ in steps(n_sweeps, "Gibbs sweeps"):
        for mask in (colour, ~colour):
            E = equation.local_field(tensors.one_hot(s, q), h, J)  # (N, L, q)
            u = torch.rand(E.shape, generator=gen, device=h.device).clamp_min(1e-12)
            s = torch.where(mask, torch.argmax(E - torch.log(-torch.log(u)), dim=-1), s)
    return s


def mask_tokens(seqs: torch.Tensor, rate: float, q: int, gen: torch.Generator):
    """Hide a random fraction of positions behind the [MASK] token q. Returns (tokens, mask) both (N, L)."""
    mask = torch.rand(seqs.shape, generator=gen, device=seqs.device) < rate
    return torch.where(mask, torch.full_like(seqs, q), seqs), mask.float()


def baselines(train: torch.Tensor, test: torch.Tensor, q: int) -> tuple[float, float]:
    """Cross-entropy (nats) on the test letters of two context-free predictors fitted on the training set:
    the unigram (one letter distribution for all sites) and the profile (one distribution per site, like a PSSM)."""
    x_train, x_test = tensors.one_hot(train, q), tensors.one_hot(test, q)
    unigram = (x_train.mean((0, 1)) * len(train) + 1) / (len(train) + q)  # (q,) add-one smoothing
    profile = (x_train.sum(0) + 1) / (len(train) + q)  # (L, q)
    ce_uni = -(x_test * torch.log(unigram)).sum(-1).mean()
    ce_prof = -(x_test * torch.log(profile)).sum(-1).mean()
    return ce_uni.item(), ce_prof.item()


def train(seqs, test, contacts, w, q, n_steps, batch, lr, rate, n_snapshots=60, seed=0):
    """Adam on the masked-LM loss. Every few steps: test loss (fixed masks), the attention contact map from the
    unmasked test sequences, and its precision at the top-K pairs (K = number of true contacts).

    Returns a dict of histories: train loss per step, and at the snapshot steps the test loss, precision,
    contact scores (S, L, L) and head-averaged first-layer attention (S, L, L).
    """
    gen = torch.Generator(device=seqs.device).manual_seed(seed)
    opt = torch.optim.Adam(tensors.parameters(w), lr=lr)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda t: min(1.0, (t + 1) / 100) * 0.5 * (1 + math.cos(math.pi * t / n_steps))
    )
    test_tokens, test_mask = mask_tokens(test, rate, q, torch.Generator(device=seqs.device).manual_seed(seed + 1))
    k_top = int(torch.triu(contacts, 1).sum())
    at = sorted({round(k * (n_steps - 1) / max(1, n_snapshots - 1)) for k in range(n_snapshots)})
    hist = {"train": torch.zeros(n_steps), "at": at, "test": [], "precision": [], "scores": [], "attention": []}
    for t in steps(n_steps, "training steps"):
        idx = torch.randint(len(seqs), (batch,), generator=gen, device=seqs.device)
        tokens, mask = mask_tokens(seqs[idx], rate, q, gen)
        logits, _ = equation.forward(tokens, w)
        loss = equation.masked_lm_loss(logits, seqs[idx], mask)
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()
        hist["train"][t] = loss.item()
        if t in at:
            with torch.no_grad():
                logits, _ = equation.forward(test_tokens, w)
                hist["test"].append(equation.masked_lm_loss(logits, test, test_mask).item())
                _, maps = equation.forward(test, w)  # unmasked sequences, as in Rao et al.
                score = equation.attention_contacts(maps)
                hist["precision"].append(equation.precision(equation.top_pairs(score, k_top), contacts))
                hist["scores"].append(score.cpu())
                hist["attention"].append(maps[:, 0].mean((0, 1)).cpu())
    hist["scores"], hist["attention"] = torch.stack(hist["scores"]), torch.stack(hist["attention"])
    return hist


def random_precision(contacts: torch.Tensor, min_sep: int = 3) -> float:
    """Expected precision of random pairs: true contacts / all pairs with j − i ≥ min_sep."""
    L = contacts.shape[0]
    i, j = torch.triu_indices(L, L, min_sep, device=contacts.device)
    return contacts[i, j].float().mean().item()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/bioinformatics/protein_language_model/plot.py
"""Animate training: the loss falls below the context-free baselines while the attention map sharpens into contacts."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .equation import top_pairs


def animate(hist: dict, contacts: torch.Tensor, n_top: int, ce_uni: float, ce_prof: float, rand: float, path: Path):
    """hist from solve.train; contacts (L, L) bool; n_top predicted pairs are marked."""
    at = np.asarray(hist["at"])
    train = hist["train"].numpy()
    test, prec = np.asarray(hist["test"]), np.asarray(hist["precision"])
    scores, attn = hist["scores"].numpy(), hist["attention"].numpy()  # (S, L, L) each
    true = contacts.cpu().numpy()
    L = true.shape[0]
    win = max(1, len(train) // 40)
    smooth = np.convolve(train, np.ones(win) / win, mode="valid")  # running mean of the noisy batch loss
    ti, tj = np.nonzero(np.triu(true, 1))

    fig, axes = plt.subplots(2, 2, figsize=(8.8, 6.2))
    (ax_loss, ax_prec), (ax_map, ax_att) = axes

    smooth_steps = np.arange(len(smooth)) + win - 1
    (train_line,) = ax_loss.plot([], [], color="#93c5fd", lw=1, label="train (batch)")
    (test_line,) = ax_loss.plot([], [], color=ACCENT[0], lw=2, label="test")
    ax_loss.axhline(ce_uni, color=ACCENT[1], ls="--", lw=1.2, label="unigram")
    ax_loss.axhline(ce_prof, color=ACCENT[4], ls="--", lw=1.2, label="per-site profile")
    ax_loss.set_xlim(0, at[-1])
    ax_loss.set_ylim(min(test.min(), ce_prof) - 0.1, ce_uni + 0.35)
    ax_loss.set_xlabel("training step")
    ax_loss.set_ylabel("masked-token loss (nats)")
    ax_loss.set_title("masked language modelling")
    ax_loss.legend(fontsize=7, loc="upper right")
    loss_dot = ax_loss.axvline(0, color="#f59e0b", lw=1)

    (prec_line,) = ax_prec.plot([], [], "o-", color=ACCENT[2], ms=3, lw=1.5, label="attention (APC)")
    ax_prec.axhline(rand, color="#9ca3af", ls="--", lw=1.2, label=f"random pairs ({rand:.2f})")
    ax_prec.set_xlim(0, at[-1])
    ax_prec.set_ylim(0, 1.05)
    ax_prec.set_xlabel("training step")
    ax_prec.set_ylabel(f"precision of top {n_top} pairs")
    ax_prec.set_title("contacts recovered from attention")
    ax_prec.legend(fontsize=7, loc="center right")

    cmap = plt.get_cmap("Blues", 12)
    for ax in (ax_map, ax_att):
        ax.set_xticks([0, L - 1], ["1", str(L)])
        ax.set_yticks([0, L - 1], ["1", str(L)])
        ax.grid(False)
    im_map = ax_map.imshow(np.zeros((L, L)), cmap=cmap, vmin=0, vmax=1)
    ax_map.scatter(tj, ti, s=28, facecolors="none", edgecolors="#374151", lw=1, label="planted contact")
    hits = ax_map.scatter([], [], s=12, color=ACCENT[2], label="top pair: hit")
    miss = ax_map.scatter([], [], s=12, color=ACCENT[1], marker="x", label="top pair: miss")
    ax_map.legend(fontsize=6.5, loc="upper right", framealpha=0.9)
    ax_map.set_title("APC attention score (lower) vs contacts (upper)", fontsize=9)
    im_att = ax_att.imshow(attn[0], cmap=plt.get_cmap("Purples", 12), vmin=0, vmax=np.percentile(attn[-1], 99))
    ax_att.set_xlabel("attended position $j$")
    ax_att.set_ylabel("query position $i$")
    title_att = ax_att.set_title("")

    def update(k: int):
        s = scores[k]
        off = s[~np.eye(L, dtype=bool)]
        lo, hi = np.percentile(off, 50), max(off.max(), 1e-9)
        shown = np.clip((s - lo) / (hi - lo), 0, 1)
        im_map.set_data(np.tril(shown, -1))  # score in the lower triangle, upper triangle left for the contacts
        pairs = top_pairs(torch.from_numpy(s), n_top).numpy()
        good = true[pairs[:, 0], pairs[:, 1]]
        hits.set_offsets(np.column_stack([pairs[good, 0], pairs[good, 1]]) if good.any() else np.empty((0, 2)))
        miss.set_offsets(np.column_stack([pairs[~good, 0], pairs[~good, 1]]) if (~good).any() else np.empty((0, 2)))
        im_att.set_data(attn[k])
        title_att.set_text(f"layer-1 attention $A_{{ij}}$, mean over heads (step {at[k]})")
        test_line.set_data(at[: k + 1], test[: k + 1])
        seen = smooth_steps <= at[k]
        train_line.set_data(smooth_steps[seen], smooth[seen])
        prec_line.set_data(at[: k + 1], prec[: k + 1])
        loss_dot.set_xdata([at[k], at[k]])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(at), 40), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'coupling': 1.2,
    'n_seq': 4000,
    'q': 8,
    'layers': 2,
    'heads': 4,
    'd_model': 32,
    'steps': 800,
    'lr': 0.003,
    'mask_rate': 0.15
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.bioinformatics.protein_language_model")]:
    del sys.modules[name]
main = importlib.import_module("problems.bioinformatics.protein_language_model.main")
problem_dir = Path("problems/bioinformatics/protein_language_model")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
